# 01 · Treinamento SFT replicado (v4)

**Ambiente: GPU (T4 ou superior).** Executar tudo. Requer o notebook 00 concluído, triagem DEV
aprovada e registro público declarado (exceto no teste técnico).

Treina 3 condições (explicit, direct, generic) × 2 níveis (A, C) × 20 sementes = 120 treinos de
120 passos (LoRA fp16). Cada sessão conclui até `MAX_JOBS_PER_SESSION` treinos; reexecute para
continuar. Treinos concluídos não são repetidos; treinos interrompidos retomam do último checkpoint.


In [1]:
# Configuração operacional. Sementes, tamanhos, modelos e critérios estão fixos no protocolo v4.
STUDY_ID = "estudo-prospectivo-2026-v4"  # @param {type:"string"}
STORAGE_MODE = "local"  # @param ["auto", "local"]
TECHNICAL_TEST = True  # @param {type:"boolean"}
MAX_JOBS_PER_SESSION = 6  # @param {type:"integer"}
CODE_COMMIT = "b858d128bcd545dd958e0dbed9978a7aea2591b3"
CODE_HASHES = {
 "__init__.py": "903b3bdb88f326693084e14633a5ebce1f862506124b58f67dc751c2a2c96f76",
 "avaliacao.py": "01b6f0f4f9e651dd2a1904eb19da2a8066a9366b976b768911722b9ada8d966f",
 "contrato.py": "fc9d9130d8edb89a56c7cf8b52dbe4ebda3b4d60d81f7964b769cb8515caa4b1",
 "estatistica.py": "7fd7615c0ead1e01c0d7e926e41c56a0b203b4ba493f3153362a9779c945bdfd",
 "modelos.py": "ee02866b8eac6db6928233805f50c985352d5c57253a3bfcf01d7c56fe910811",
 "monitor.py": "3635ed93e186e64dea87b0df428c8472434f40e2d243660b7ffb63ddce2a87f3",
 "runtime.py": "c1f4326e798b39e8b90fc6099d1c8da8b31e14a0ec9b46bf678c43c2a77fe3b0",
 "tarefas.py": "8b1a58ecc9df6b959741409f71811b7e90449bc41eedf7d860528f74732ba7fd",
 "treino.py": "357cc924119facab1911ca07a8e76c987ce5c64e79050381887fcb0d72202fed"
}
SUPPORT_HASHES = {
 "armazenamento.py": "66e04b1224550f8532311273ae78e118ffe1766654c6d0ee9c43ab2557d2482f",
 "sessao.py": "3c8c3774d66fce7fc0881bfb2db0e65d7bfbb2a2a38ef36f48373ddba29dde41"
}

import hashlib, importlib.util, os, re, subprocess, sys, time
from pathlib import Path
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID aceita somente letras, números, _ e -.")
if TECHNICAL_TEST and not STUDY_ID.endswith("-teste-tecnico"):
    STUDY_ID += "-teste-tecnico"  # pasta separada: nunca se mistura ao estudo
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    drive, ON_COLAB = None, False
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT", "/content/cot-v4" if ON_COLAB else Path.cwd() / "cot-v4"))
LOCAL.mkdir(parents=True, exist_ok=True)

def checkout(ref):
    """Obtém exatamente o commit fixado; não usa clones antigos nem a branch atual."""
    override = os.environ.get("COT_REPO_DIR")
    if override:
        return Path(override).resolve()
    repo = LOCAL / "source" / ref
    git = lambda *a, **k: subprocess.run(["git", "-C", str(repo), *a], capture_output=True, text=True, **k)
    if not (repo / ".git").is_dir():
        repo.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet", str(repo)], check=True)
        git("remote", "add", "origin", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git")
    if git("cat-file", "-e", ref + "^{commit}").returncode:
        for attempt in range(1, 4):
            fetched = git("fetch", "--quiet", "--depth", "1", "origin", ref)
            if not fetched.returncode:
                break
            print(f"Falha de rede ao baixar o código (tentativa {attempt}/3): {fetched.stderr.strip()[-300:]}")
            time.sleep(5 * attempt)
        else:
            raise RuntimeError("Não foi possível baixar o commit " + ref + ". Verifique a conexão e execute de novo.")
    if git("checkout", "--quiet", "--detach", ref).returncode:
        raise RuntimeError("Checkout do commit fixado falhou: " + ref)
    if git("rev-parse", "HEAD").stdout.strip() != ref:
        raise RuntimeError("O código aberto não corresponde ao commit do notebook.")
    return repo

REPO = checkout(CODE_COMMIT)
PACKAGE = REPO / "experimento_cientifico_v4"
def load(name):
    path = PACKAGE / "colab" / name
    if hashlib.sha256(path.read_bytes()).hexdigest() != SUPPORT_HASHES[name]:
        raise RuntimeError("Suporte divergente do notebook: " + name + ". Reabra pelo link do GitHub.")
    # Nome único por commit: módulos de versões antigas na mesma sessão não são reutilizados.
    spec = importlib.util.spec_from_file_location(f"cotv4_{name[:-3]}_{CODE_COMMIT[:12]}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
armazenamento, sessao = load("armazenamento.py"), load("sessao.py")
STATE = armazenamento.prepare_storage(LOCAL, STUDY_ID, on_colab=ON_COLAB, mode=STORAGE_MODE,
    storage_override=os.environ.get("COT_STORAGE_ROOT"), drive_module=drive,
    mountpoint=os.environ.get("COT_DRIVE_MOUNTPOINT", "/content/drive"))
STUDY = sessao.Session(REPO, STATE, CODE_COMMIT, CODE_HASHES, on_colab=ON_COLAB, technical_test=TECHNICAL_TEST)
Blocked = sessao.StageBlocked
print("Código:", CODE_COMMIT, "| Estudo:", STUDY_ID, "| Teste técnico:", TECHNICAL_TEST)


MODO LOCAL TEMPORÁRIO: /content/cot-v4/studies/estudo-prospectivo-2026-v4-teste-tecnico


Os arquivos desta pasta são apagados quando a sessão do Colab termina.


Nesse modo, o notebook 00 prepara e exporta o pacote, e treinos e avaliações do estudo ficam pausados; só o teste técnico (TECHNICAL_TEST=True) roda em pasta temporária.


  - STORAGE_MODE="local": o Drive não foi solicitado.


Para reconectar: mantenha STORAGE_MODE="auto", execute de novo a primeira célula e autorize a conta. Se o erro persistir, use Ambiente de execução > Desconectar e excluir ambiente de execução e abra o notebook novamente pelo link do GitHub.


Código: b858d128bcd545dd958e0dbed9978a7aea2591b3 | Estudo: estudo-prospectivo-2026-v4-teste-tecnico | Teste técnico: True


In [2]:
status = STUDY.execute("status")
gate_ok = status["prepared"] and (status["gate"]["passed"] or TECHNICAL_TEST)
registered = status["registration"]["passed"] or TECHNICAL_TEST
if not status["prepared"]:
    print("ETAPA BLOQUEADA: execute primeiro o notebook 00 com o mesmo STUDY_ID.")
elif not gate_ok:
    print("ETAPA BLOQUEADA:", sessao.explain(status["gate"].get("reasons", [])))
elif not registered:
    print("ETAPA BLOQUEADA: declare o registro público no notebook 00 antes de treinar.")
else:
    try:
        progress = STUDY.execute("train", mode="sft", max_jobs=MAX_JOBS_PER_SESSION)
        print(f"SFT: {progress['completed']}/{progress['total']} treinos concluídos ({progress['status']}).")
    except Blocked as reason:
        print("ETAPA BLOQUEADA:", reason)
STUDY.export(download=False)
STUDY.status()


Etapa "status" concluída; resultado completo em /content/cot-v4/studies/estudo-prospectivo-2026-v4-teste-tecnico/operations/status_sft.json.


TESTE TÉCNICO em pasta temporária: "train" será perdido ao fim da sessão e não constitui evidência científica.


Preparando ambiente isolado (uma vez por sessão). O PyTorch do Colab é preservado.


venv indisponível neste Python (sem ensurepip); usando virtualenv isolado.


created virtual environment CPython3.13.16.final.0-64 in 243ms


...

SFT: 6/6 treinos concluídos (all_training_complete).
ZIP salvo: /content/cot-v4/studies/estudo-prospectivo-2026-v4-teste-tecnico/downloads/resultados_v4_TESTE_TECNICO.zip (87 KiB, SHA256 2bb1bb44e80745d2…)


Atenção: o ZIP está em pasta temporária; baixe-o antes de encerrar a sessão.


Etapa "status" concluída; resultado completo em /content/cot-v4/studies/estudo-prospectivo-2026-v4-teste-tecnico/operations/status_sft.json.



SITUAÇÃO DO ESTUDO — TESTE TÉCNICO (não é evidência científica)
  00 Protocolo e dados         CONCLUÍDA        
  Triagem DEV (65% por nível)  CONCLUÍDA        sft/A=97.5% sft/C=68.3% grpo/low=95.0% grpo/high=90.8%
  Registro público             NÃO NECESSÁRIO (teste técnico) external_registration_missing
  01 Treinos SFT               CONCLUÍDA        6/6
  02 Treinos GRPO              EXCLUÍDA         braço excluído antes do registro (ver protocolo)
  03 Avaliação TEST            EM ANDAMENTO     0/6
  Armazenamento: /content/cot-v4/studies/estudo-prospectivo-2026-v4-teste-tecnico | persistente: False


[('00 Protocolo e dados', 'CONCLUÍDA', ''),
 ('Triagem DEV (65% por nível)',
  'CONCLUÍDA',
  'sft/A=97.5% sft/C=68.3% grpo/low=95.0% grpo/high=90.8%'),
 ('Registro público',
  'NÃO NECESSÁRIO (teste técnico)',
  'external_registration_missing'),
 ('01 Treinos SFT', 'CONCLUÍDA', '6/6'),
 ('02 Treinos GRPO',
  'EXCLUÍDA',
  'braço excluído antes do registro (ver protocolo)'),
 ('03 Avaliação TEST', 'EM ANDAMENTO', '0/6')]

**Situação das etapas.** CONCLUÍDA: arquivos verificados por hash. BLOQUEADA: um critério do
protocolo ou um recurso impede a etapa (o motivo é impresso). NÃO EXECUTADA / PENDENTE: ainda
não iniciada. Reexecutar retoma o mesmo `STUDY_ID`; trocar o ID inicia outro estudo.
Cópias antigas salvas no Colab mantêm o commit antigo: abra sempre pelo link do README.
